# Assignment 2 — results and your step-by-step guide

Built from the saved outputs on 2026-09-26. Every number below is read from files in `Claude/03_Project/outputs/` and `Claude/04_Project/outputs/week4_validation/`.

**Due:** PDF by **Sunday Sep 27, 2026**. Presentation **Monday Sep 28, 2026** (4–5 minutes).

Sections: 1 Results at a glance · 2 Evidence file index · 3 What *you* must do · 4 Honor-code note

## 1. Results at a glance

### 1.1 Route comparison (original run, 50 m gap tolerance)

Source: `Claude/03_Project/outputs/default/route_comparison.csv`. Rate is an invented classroom value (USD 10/m).

| Route | Status | Length (m) | Connectors (count / total m / max m) | Cost (USD) |
|---|---|---|---|---|
| baseline | `ok` | 22,896.8 | 0 / 0.0 / 0.0 | 228,967.56 |
| roads | `ok` | 22,897.6 | 2 / 1.3 / 0.7 | 228,975.93 |
| power | `endpoint_too_far` | — | — | — |
| rail | `ok` | 25,902.4 | 2 / 567.5 / 372.7 | 259,023.67 |

Every connector used on a route is an **endpoint connector** (roads: 0.59 m + 0.72 m; rail: 194.8 m + 372.7 m). No route uses a gap connector.

### 1.2 Audit check counts

Source: `validation_table.csv` (75 checks).

| PASS | FAIL | NOT_CHECKED |
|---|---|---|
| 71 | 1 | 3 |

- The **FAIL** is `V-crossings`, a real finding (see 1.4).
- The three **NOT_CHECKED** rows are the external-evidence sites. They wait for *your* imagery check (step 3c).

### 1.3 Sensitivity: only the gap tolerance changed

Endpoint limit (500 m), endpoints, study rectangle, input data and rate are fixed in every run. The audit's `S-*-ctrl` checks confirm that only the gap setting differs.

| Gap tolerance | baseline | roads | power | rail | Power components after repair | Ranking |
|---|---|---|---|---|---|---|
| 25 m | 22,896.8 m (0 conn.) | 22,897.6 m (2 conn.) | `endpoint_too_far` | 25,902.4 m (2 conn.) | 6 (of 9) | baseline < roads < rail |
| 50 m (original) | 22,896.8 m (0 conn.) | 22,897.6 m (2 conn.) | `endpoint_too_far` | 25,902.4 m (2 conn.) | 3 (of 9) | baseline < roads < rail |
| 250 m | 22,896.8 m (0 conn.) | 22,897.6 m (2 conn.) | `endpoint_too_far` | 25,902.4 m (2 conn.) | 1 (of 9) | baseline < roads < rail |

**Reading:** route availability, lengths, connectors and ranking are identical at 25, 50 and 250 m. Only the power layer's internal fragmentation responds to the threshold. Power stays unavailable because its *endpoint* is too far, not because of internal gaps.

### 1.4 Findings worth presenting

| # | Finding | Evidence |
|---|---|---|
| 1 | **Rail route uses an unverified 2D crossing onto a bridge.** It transfers from OSM way **891041250** (main line, not a bridge) to way **239559205** (a 36 m bridge tagged `layer=1`) at **lat -3.993745, lon 39.54626**. Neither line has a vertex there (nearest vertices are 21 m and 15 m away), so the model joined two lines that may be grade-separated. | `V-crossings` (FAIL), `inferred_crossings.gpkg`, red × on `validation_map.png` |
| 2 | **An error in the Week 3 discussion answer.** Q1 said rail used one gap connector and one endpoint connector. `connectors.gpkg` shows both are `endpoint_connector` (194.8 m + 372.7 m = 567.5 m). It has been corrected, with a note. | `Claude/03_Project/outputs/default/connectors.gpkg`, `discussion_answers.ipynb` |
| 3 | **A bug in the self-check.** The routing script's own connector-limit check hard-coded 50 m, so it ignored the gap setting under reruns. It has been fixed to use the actual tolerance. The routing agent's self-validation had a blind spot. | `week3_routing.py` (`validate_result`) |
| 4 | **Power `endpoint_too_far` confirmed independently.** start 1839.2 m, end 995.1 m from nearest power line, against the 500 m limit. | `U-power` |

Other checks worth a sentence: UTM lengths are within 0.04% of geodesic lengths (metres, not degrees). The non-connector length of each route lies on mapped lines (`R-*-mapped`). The 2,290 road features dropped are OSMnx reversed duplicates, which is expected.

### 1.5 Conclusions and labels (from `validation_report.md`)

- Route lengths, endpoints, containment and costs are computed correctly — **VERIFIED** (R-*, C-*).
- Connectors are within their limits and counted once; every used connector is an endpoint connector — **VERIFIED** (R-*-conn*).
- Power is unavailable because its start point is far from any mapped power line, not because of the gap threshold — **VERIFIED** (U-power, S-*); whether a real line exists there is **UNRESOLVED** without utility data.
- Availability and ranking (baseline < roads < rail) do not change between 25, 50 and 250 m gap tolerance — **not SENSITIVE** to this one setting (S-*). This is a single setting rerun, not a global robustness test.
- The roads route follows a buildable corridor through every junction — **UNRESOLVED**: 1 transfer(s) happen at unverified 2D crossings (see map); these need imagery.
- Connector sites and crossings match reality on the ground — **UNRESOLVED** (E-*).
- Access rights, permits and construction feasibility along any corridor — **UNRESOLVED**; mapped proximity does not establish them, and no legal or engineering evidence was used.
- Costs are relative, invented classroom values (USD 10/m) — not a construction estimate.


## 2. Evidence file index

The assignment asks you to retain these files and identify them in your supporting slides.

| Item | Path |
|---|---|
| Routing script (Week 3) | `Claude/03_Project/week3_routing.py` |
| Audit script | `Claude/04_Project/week4_audit.py` |
| Run commands | `python week3_routing.py` · `python week3_routing.py --gap-tolerance-m 25 --out outputs/sensitivity_gap_25m` · `python week3_routing.py --gap-tolerance-m 250 --out outputs/gap_tolerance_250m` · `python week4_audit.py` |
| Package versions | Python 3.12.2; geopandas 1.1.4; shapely 2.1.2; pyproj 3.8.0; networkx 3.6.1; pandas 2.3.3 (recorded in every `routing_report.md` and in `validation_report.md`) |
| Original run | `Claude/03_Project/outputs/default/` (`routes.gpkg`, `connectors.gpkg`, `route_comparison.csv`, `route_map.png`, `routing_report.md`) |
| Sensitivity rerun (required, 25 m) | `Claude/03_Project/outputs/sensitivity_gap_25m/` |
| Extra rerun (250 m, Week 3) | `Claude/03_Project/outputs/gap_tolerance_250m/` |
| Check table | `Claude/04_Project/outputs/week4_validation/validation_table.csv` |
| Validation report | `Claude/04_Project/outputs/week4_validation/validation_report.md` |
| Maps | `…/week4_validation/validation_map.png` (audit) · `…/default/route_map.png` (routes) |
| Flagged crossings | `…/week4_validation/inferred_crossings.gpkg` |
| External evidence | `…/week4_validation/external_evidence.csv` |

**Location note:** 04_02 names `outputs/week3_routing/` and `outputs/week4_validation/` at the repo root. These files live under `Claude/` so the course repo stays read-only. Do **one** of the following:

- State the location on your evidence-index slide ("outputs kept under `Claude/…` to preserve the course repo"), **or**
- Copy them into the named folders (run from `Agentic-GeoAI/`):

```bash
mkdir -p outputs/week3_routing outputs/week4_validation
cp -R Claude/03_Project/outputs/* outputs/week3_routing/
cp -R Claude/04_Project/outputs/week4_validation/* outputs/week4_validation/
cp Claude/03_Project/week3_routing.py outputs/week3_routing/
cp Claude/04_Project/week4_audit.py outputs/week4_validation/
```

## 3. Step-by-step: what YOU must do

Total: about **2½–3 hours**.

### a. Read the results (15 min)
Read section 1 above. Open `Claude/04_Project/outputs/week4_validation/validation_map.png` and find the red × (the rail crossing) and the lime dashed connectors. Skim `validation_table.csv`.

### b. Write your own expected checks (20 min)
04_02 section 13 wants these written **by you**, as if before the audit. Use your own words and at least 5 rows.

| Check | Expected result / tolerance | Evidence | What failure would mean |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

* Claim geometry cannot establish: …

*Hints (pick and phrase yourself):* endpoints, continuity, geometry validity, study-area containment, units/CRS, connector limits and counting once, cost arithmetic, blank rows for unavailable routes, route ≥ straight-line distance, crossings/bridges, access rights.

### c. Imagery check of the three sites (30–40 min)
This turns the three NOT_CHECKED rows into real external evidence and is your strongest slide.

| Site | Lat, lon | Claim to test |
|---|---|---|
| `rail_endpoint_connector_3` | -4.008849, 39.591099 | No mapped rail line closer than 195 m to the endpoint (connector is new construction, not a missing segment) |
| `rail_endpoint_connector_4` | -3.864413, 39.472359 | No mapped rail line closer than 373 m to the endpoint (connector is new construction, not a missing segment) |
| `rail_crossing_1` | -3.993745, 39.54626 | rail route transfers 891041250 -> 239559205 at grade (no bridge/tunnel separating them) |

**What to look for**
- **`rail_crossing_1`:** where the two rail lines meet, does one pass **over** the other on a bridge (grade separation) or do they join at the same level? A bridge or overpass means the model's "2D crossings connect" assumption is wrong here, so answer `supports_claim = no`.
- **Endpoint connectors (195 m and 373 m):** is there really **no rail track** closer to the endpoint than that distance? Unmapped sidings or spurs would mean OSM is incomplete there.

**How**
- **Google Earth Pro** (free desktop app): paste `lat, lon` into Search. Use the clock icon (Historical Imagery) to see the image date and record it.
- **Esri World Imagery Wayback** (livingatlas.arcgis.com/wayback): search the coordinates and pick a release; the release date is the imagery date.
- Take a screenshot of each site for your slides.

**Fill in `Claude/04_Project/outputs/week4_validation/external_evidence.csv`** (Excel or a text editor; keep the header row):

| Column | What to write |
|---|---|
| `site_id`, `claim`, `lon`, `lat` | leave as they are |
| `source` | e.g. `Esri World Imagery Wayback` or `Google Earth Pro (Maxar)` |
| `source_date` | image date, e.g. `2024-03-15` |
| `provenance` | who captured it and the release or version |
| `independent_of_osm` | `yes` or `no`. Imagery is `yes`; any OSM-based map is `no` |
| `observation` | one sentence on what you saw |
| `supports_claim` | `yes`, `no`, or `partly` |

**Rerun** (from `Agentic-GeoAI/`):

```bash
/opt/anaconda3/bin/python Claude/04_Project/week4_audit.py
```

What should change: each filled `E-*` row becomes **PASS** (supports `yes`) or **FAIL** (`no` or `partly`). If all three pass, the "connector sites and crossings" conclusion changes from UNRESOLVED to EXTERNALLY SUPPORTED. A FAIL is a valid finding, so don't hide it. Rows marked `independent_of_osm = no` stay NOT_CHECKED.

### d. Screenshots into the external-evidence slide (10 min)
Paste each site's imagery screenshot into the draft deck's external-evidence slide. Label each with its site ID, the source and the image date.

### e. AI-use statement (10 min)
Rewrite the draft AI-use statement **in your own words**. It must name the tools (Claude Code, Opus 5.5) and what they did: wrote and debugged the routing and audit scripts, ran the reruns and drafted the slides. It must also say what *you* did: specified the checks, did the imagery review, verified the results and edited the deck.

### f. Adapt, rehearse, export, submit (60–75 min)
1. Adapt the draft deck. Keep the framework intro short (about 1 minute); most of the time goes to evidence.
2. Make sure the deck has the map with distinct connectors **and** OSM attribution, a readable check summary, the sensitivity table, sources and the AI-use statement.
3. Rehearse with a timer and aim for 4:30. Move detail into supporting slides.
4. Export to **PDF** and submit by **Sunday Sep 27, 2026**. Present on **Monday Sep 28**.

### g. Be ready to explain the code (20 min)

| Likely question | Short answer |
|---|---|
| Why EPSG:32737? | It's UTM zone 37S, a projected CRS in metres covering the Mombasa corridor (36–42°E). Lengths and thresholds need metres, not degrees. The audit shows UTM vs geodesic length differs by < 0.04%. |
| How does gap repair work? | It builds a graph per layer and finds the closest pair of disconnected components. If their gap is ≤ the tolerance and inside the study area, it adds a straight `gap_connector`, then repeats. Larger gaps are never bridged. |
| Why is the audit (self-check) not independent validation? | The same AI wrote the routing code and the audit, so they can share assumptions. It verifies computations; real-world claims need independent evidence such as imagery. |
| Why can't the cost rate change the ranking? | Cost = length × one common positive rate, so scaling every length by the same k keeps their order. |
| Why is power unavailable, and would a bigger gap tolerance fix it? | Its start point is 1,839 m from any mapped power line, beyond the 500 m endpoint limit. Gap tolerance only joins pieces *within* the layer. At 250 m the power layer becomes 1 component and it is still unavailable. |

### One decision for a human, one feedback example (the assignment asks for both)
- **Human decision:** whether a route is acceptable to pursue, e.g. accepting new construction at the rail endpoint connectors, or judging access rights and permits. That's an institutional or legal call, not geometry.
- **Feedback that should change the agent's next action:** a failed check such as `V-crossings` (a transfer at a bridge without a shared node). The agent should stop treating that crossing as connected, e.g. forbid it and re-route, or report the route as conditional, rather than raise tolerances to force success.

## 4. Honor-code note

The course AI policy allows AI tools, but **you** remain responsible for understanding, testing, validating and documenting everything you submit. Before presenting:

- Be able to explain every script step and every number on your slides (section 3g helps).
- Disclose AI use accurately in the AI-use statement.
- Don't present imagery findings you didn't check yourself, and cite real sources only (OSM, Esri, Google, and the 04_01/04_02 readings you actually use).

Undisclosed AI material or code you can't explain may be reported as an Honor Code violation.